In [1]:
import pandas as pd
import numpy as np
import json

/n--- Translating EN to AR ---


In [11]:
df_translation.head()

,col_en,val_en,col_ar,val_ar
0,Age Group,0-1 years,الفئة العمرية,0-1 سنة
1,Age Group,0-4 years,الفئة العمرية,0-4 سنوات
2,Age Group,10-14 years,الفئة العمرية,10-14 سنة
3,Age Group,1-4 years,الفئة العمرية,1-4 سنوات
4,Age Group,15+ years,الفئة العمرية,15+ سنة


In [ ]:
print("/n--- Translating EN to AR ---")
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR'
df_translation=pd.read_excel(path+'/translation dict.xlsx')


En_Ar_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_en'].unique()) - {'Year', 'Value', 'Source'}

for dim in dimensions:
    df_dim=df_translation[df_translation['col_en'].isin([dim.lower(), dim])].copy()
    En_Ar_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_en'], df_dim['val_ar'])), 
                'dim': {df_dim['col_en'].unique()[0]:df_dim['col_ar'].unique()[0]}}})

Ar_En_dictionary={}
#get the unique dimensions
dimensions = set(df_translation['col_en'].unique()) - {'السنة', 'العدد', 'المصدر'}

for dim in dimensions:
    df_dim=df_translation[df_translation['col_ar'].isin([dim.lower(), dim])].copy()
    Ar_En_dictionary.update(
        {dim:{'dim_values':dict(zip(df_dim['val_ar'], df_dim['val_en'])), 
                'dim': {df_dim['col_ar'].unique()[0]:df_dim['col_en'].unique()[0]}}})

class Translator:
    def __init__(self, translate_to, en_ar_dict, ar_en_dict):
        self.translate_to = translate_to.lower()
        self.en_ar_dict = en_ar_dict
        self.ar_en_dict = ar_en_dict

    def translate(self, df):
        df_translated = df.copy()
        
        # Pick the dictionary based on direction
        mapping = self.ar_en_dict if self.translate_to == 'english' else self.en_ar_dict
        
        for col, col_dict in mapping.items():
            if col in df_translated.columns:
                #Replace the data values (e.g., 'Male' -> 'ذكر')
                df_translated[col] = df_translated[col].replace(col_dict['dim_values'])
                
                #Rename the column header (e.g., 'Gender' -> 'الجنس')
                df_translated.rename(columns=col_dict['dim'], inplace=True)
                
        return df_translated

In [ ]:
#read in the dataframe
path='C:/Users/raffi/OneDrive - United Nations/Desktop/DSS/DATA COLLECTOR/datacollector_received_quest/EN quests'
merged_en_df=pd.read_excel(path+'/yourfolder/yourfile.xlsx')

translator = Translator('arabic')
en_df_translated = translator.translate(merged_en_df)

# final_df = pd.concat([merged_ar_df, en_df_translated], ignore_index=True)